# LAB 3: NLP Subtasks Using Hugging Face Transformer Models

**Mapped COs:** CO2, CO4

## Aim
To understand Hugging Face and demonstrate common NLP subtasks using pretrained Transformer models through a simple interactive user interface.

## Hugging Face tasks studied
- Text classification / sentiment analysis
- Text summarization
- Question answering
- Token classification / Named Entity Recognition (NER)
- Text generation
- Translation

## Theory
**Hugging Face** provides libraries, pretrained models, datasets, and tools for machine learning and natural language processing.

A **Transformer** is a neural-network architecture that uses attention mechanisms to model relationships between tokens in a sequence.

A **pipeline** is a high-level Transformers API that combines preprocessing, model inference, and postprocessing for a task.

The **Hugging Face Hub** hosts models, datasets, and demo applications. **Hugging Face Spaces** hosts interactive machine-learning demos.

## Pseudocode
1. Import Transformers and Gradio.
2. Create a UI for task selection and text input.
3. Load a pretrained pipeline for the selected NLP task.
4. Pass user input to the pipeline.
5. Display formatted results and handle invalid input or model errors.
6. Repeat for additional user inputs.

## Cell 1 — Install dependencies

In [ ]:
%pip install -q transformers gradio torch sentencepiece

## Cell 2 — Import libraries and define available tasks

In [ ]:
import gradio as gr
from transformers import pipeline
import torch
import traceback

DEVICE = 0 if torch.cuda.is_available() else -1
print("PyTorch:", torch.__version__)
print("Device:", "GPU" if DEVICE == 0 else "CPU")
print("Gradio:", gr.__version__)

## Cell 3 — Create the NLP processing function

Each task uses a pretrained model from the Hugging Face Hub. Models are loaded on first use and cached for later requests. Translation uses an English-to-French model; question answering requires both a context paragraph and a question.

In [ ]:
MODEL_CONFIG = {
    "Sentiment Analysis": {
        "task": "sentiment-analysis",
        "model": "distilbert/distilbert-base-uncased-finetuned-sst-2-english",
    },
    "Text Summarization": {
        "task": "summarization",
        "model": "sshleifer/distilbart-cnn-12-6",
    },
    "Question Answering": {
        "task": "question-answering",
        "model": "distilbert/distilbert-base-cased-distilled-squad",
    },
    "Named Entity Recognition (NER)": {
        "task": "ner",
        "model": "dslim/bert-base-NER",
    },
    "Text Generation": {
        "task": "text-generation",
        "model": "distilgpt2",
    },
    "Translation (English to French)": {
        "task": "translation",
        "model": "Helsinki-NLP/opus-mt-en-fr",
    },
}

_pipeline_cache = {}

def get_pipeline(task_name):
    config = MODEL_CONFIG[task_name]
    key = (config["task"], config["model"])
    if key not in _pipeline_cache:
        _pipeline_cache[key] = pipeline(
            task=config["task"],
            model=config["model"],
            device=DEVICE
        )
    return _pipeline_cache[key]


def run_nlp(task_name, text, question=""):
    if not text or not text.strip():
        return "Please enter some text."

    try:
        nlp = get_pipeline(task_name)

        if task_name == "Sentiment Analysis":
            result = nlp(text[:5000])
            return "\n".join(
                f"Label: {item['label']} | Confidence: {item['score']:.4f}"
                for item in result
            )

        elif task_name == "Text Summarization":
            if len(text.split()) < 25:
                return "Please enter a longer passage (about 25 words or more) for summarization."
            result = nlp(
                text[:5000],
                max_length=100,
                min_length=10,
                do_sample=False,
                truncation=True
            )
            return result[0]["summary_text"]

        elif task_name == "Question Answering":
            if not question or not question.strip():
                return "Enter a question in the Question field."
            result = nlp(
                question=question.strip(),
                context=text[:5000]
            )
            return (
                f"Answer: {result['answer']}\n"
                f"Confidence: {result['score']:.4f}\n"
                f"Character positions: {result['start']}–{result['end']}"
            )

        elif task_name == "Named Entity Recognition (NER)":
            result = nlp(text[:3000], aggregation_strategy="simple")
            if not result:
                return "No named entities detected."
            return "\n".join(
                f"Entity: {item['word']} | Type: {item['entity_group']} "
                f"| Confidence: {item['score']:.4f}"
                for item in result
            )

        elif task_name == "Text Generation":
            result = nlp(
                text[:1000],
                max_new_tokens=60,
                do_sample=True,
                temperature=0.8,
                pad_token_id=nlp.tokenizer.eos_token_id
            )
            return result[0]["generated_text"]

        elif task_name == "Translation (English to French)":
            result = nlp(text[:1000], max_length=200, truncation=True)
            return result[0]["translation_text"]

        return "Unknown task selected."

    except Exception as exc:
        return (
            f"Could not complete {task_name}.\n"
            f"Error: {type(exc).__name__}: {exc}\n\n"
            "Check your internet connection, model access, and installed packages. "
            "The first request may take time while downloading model files."
        )

## Cell 4 — Test a pipeline without the UI

In [ ]:
# This downloads the sentiment model on first run; internet access is required.
demo_result = run_nlp(
    "Sentiment Analysis",
    "I really enjoyed learning how Transformer models work!"
)
print(demo_result)

## Cell 5 — Build the interactive Gradio UI

In [ ]:
TASKS = list(MODEL_CONFIG.keys())

with gr.Blocks(title="Hugging Face NLP Lab") as demo:
    gr.Markdown("# Hugging Face NLP Playground")
    gr.Markdown(
        "Choose an NLP task, enter text, and click **Run NLP Task**. "
        "Models are downloaded from the Hugging Face Hub when first used."
    )

    task_dropdown = gr.Dropdown(
        choices=TASKS,
        value="Sentiment Analysis",
        label="Select NLP Task"
    )

    text_input = gr.Textbox(
        label="Input Text / Context",
        placeholder="Enter a sentence or paragraph here...",
        lines=6
    )

    question_input = gr.Textbox(
        label="Question (only for Question Answering)",
        placeholder="Ask a question about the context above...",
        lines=2
    )

    run_button = gr.Button("Run NLP Task", variant="primary")
    output = gr.Textbox(label="Model Output", lines=8, interactive=False)

    run_button.click(
        fn=run_nlp,
        inputs=[task_dropdown, text_input, question_input],
        outputs=output
    )

    gr.Examples(
        examples=[
            ["Sentiment Analysis", "This product is excellent and easy to use.", ""],
            ["Text Summarization",
             "Artificial intelligence is a field of computer science. It develops systems that perform tasks normally requiring human intelligence. Natural language processing enables computers to process, understand, and generate human language. Transformer models use attention mechanisms to learn relationships among tokens and are widely used for translation, summarization, and question answering.",
             ""],
            ["Question Answering",
             "Hugging Face is a platform that hosts machine learning models, datasets, and interactive demos. Transformers provides pretrained models for many natural language processing tasks.",
             "What does Hugging Face host?"],
            ["Named Entity Recognition (NER)",
             "Satya Nadella works at Microsoft in Redmond.", ""],
            ["Text Generation", "Artificial intelligence will", ""],
            ["Translation (English to French)", "Hello, how are you?", ""],
        ],
        inputs=[task_dropdown, text_input, question_input]
    )

demo.launch()

## Cell 6 — Optional: create a public Gradio share link

If you want to demonstrate the app to someone else, change the last line of Cell 5 to:

```python
demo.launch(share=True)
```

A temporary public link may be created. Do not enter confidential or personal information into a publicly shared demo.

## Expected observations
- Sentiment analysis returns a label and confidence score.
- Summarization produces a shorter version of a long passage.
- Question answering extracts an answer from the supplied context.
- NER identifies entities such as people, organizations, and locations.
- Text generation continues the supplied prompt.
- Translation converts English text into French.

Actual results vary by model and input. The first inference may take several minutes while model weights download.

## Result and conclusion
An interactive NLP application was designed using Hugging Face Transformers and Gradio. It demonstrates sentiment analysis, summarization, question answering, named entity recognition, text generation, and translation using pretrained models.

## FAQs
**1. What is Hugging Face?**  
A platform and ecosystem for sharing and using machine-learning models, datasets, and applications.

**2. What is a Transformer model?**  
A neural-network architecture that uses attention to learn contextual relationships between tokens.

**3. What is a Hugging Face pipeline?**  
A high-level API that simplifies input preprocessing, model inference, and output formatting for a particular task.

**4. What is Hugging Face Hub?**  
A repository for publishing and discovering models, datasets, and related artifacts.

**5. What is the purpose of Hugging Face Spaces?**  
To host and share interactive machine-learning applications and demos.

## Screenshots to include in the practical record
1. The running Gradio UI with the task dropdown.
2. Sentiment analysis output.
3. Summarization or question-answering output.
4. Named Entity Recognition output.
5. Translation or text-generation output.